In [ ]:
#!/usr/bin/env python3
"""
07c_eval_claude_colab.py — Claude Sonnet Evaluation on MedConflict-Bench
=========================================================================
Run on GOOGLE COLAB. Evaluates Claude Sonnet 4 on the benchmark.

Setup in Colab:
    !pip install anthropic tqdm scikit-learn

    # Then paste this entire file as a cell and run.
    # Set your API key below.
"""

# ═══════════════════════════════════════════════════════════════════════════
#  SET YOUR API KEY HERE
# ═══════════════════════════════════════════════════════════════════════════
from google.colab import userdata

# Safely fetch the key from the Secrets tab
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

# Example of how you would pass it to the Anthropic library:
# import anthropic
# client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Model to use
MODEL = "claude-opus-4-7"

# Set to 50 for a quick test (~$0.50), None for full eval (~$3-5)
MAX_PER_STRATUM = None
# MAX_PER_STRATUM = 50

# ═══════════════════════════════════════════════════════════════════════════

import json, os, re, time, sys
import numpy as np
from pathlib import Path
from tqdm import tqdm

# Mount Drive
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_BASE = Path("/content/drive/MyDrive/NIPS_2026/MedConflict-Bench")
    ON_COLAB = True
except Exception:
    # Running locally
    DRIVE_BASE = Path(r"C:\Users\AIML\Sai_Teja _Honors\genmedassist")
    ON_COLAB = False

BENCH_PATH = DRIVE_BASE / "medconflict_bench_v2.jsonl"
RESULTS_DIR = DRIVE_BASE / "eval_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Install anthropic if needed
try:
    import anthropic
except ImportError:
    os.system(f"{sys.executable} -m pip install anthropic --quiet")
    import anthropic

from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

# ── Prompt ─────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a clinical pharmacist reviewing a patient's medication order.
Determine if there is a clinically significant conflict between the prescribed medication and the patient's clinical data.

A conflict can be:
1. A drug prescribed to a patient with a documented allergy or contraindicated condition
2. A drug prescribed when a lab value exceeds a safety threshold
3. A drug-drug interaction that becomes dangerous given the patient's lab values

Respond with EXACTLY this JSON format and nothing else:
{"conflict_detected": true/false, "severity": "none"|"moderate"|"major", "explanation": "brief reasoning"}"""


def build_prompt(example):
    parts = ["=== MEDICATION ORDER REVIEW ===\n"]

    drug = example.get("drug", "Unknown")
    dose = example.get("dose_val", "")
    unit = example.get("dose_unit", "")
    route = example.get("route", "")

    drug_line = f"Prescribed: {drug}"
    if dose and unit and str(dose) != "nan" and str(unit) != "nan":
        drug_line += f" {dose} {unit}"
    if route and str(route) != "nan":
        drug_line += f" ({route})"
    parts.append(drug_line)

    if example.get("interacting_drug"):
        parts.append(f"Co-prescribed: {example['interacting_drug']}")

    lab_name = example.get("lab_name")
    lab_value = example.get("lab_value")
    lab_unit = example.get("lab_unit")

    if lab_name and lab_value is not None:
        lab_line = f"\nLab: {lab_name} = {lab_value}"
        if lab_unit and str(lab_unit) != "nan":
            lab_line += f" {lab_unit}"
        parts.append(lab_line)

    condition = example.get("conflict_condition", "")
    if example.get("stratum") == 1 and condition and condition != "none":
        parts.append(f"\nPatient history: {condition}")

    parts.append("\n=== END ===")
    parts.append("\nIs there a conflict? Respond with JSON only.")
    return "\n".join(parts)


def parse_response(raw_response):
    text = raw_response.lower().strip()

    json_match = re.search(r'\{[^}]+\}', raw_response)
    if json_match:
        try:
            parsed = json.loads(json_match.group())
            detected = parsed.get("conflict_detected", parsed.get("conflict", False))
            if isinstance(detected, str):
                detected = detected.lower() in ["true", "yes", "1"]
            return bool(detected)
        except json.JSONDecodeError:
            pass

    safe_signals = ["no conflict", "safe to proceed", "no clinically significant",
                    "\"false\"", "no significant", "conflict_detected\": false"]
    for s in safe_signals:
        if s in text:
            return False

    conflict_signals = ["conflict_detected\": true", "conflict detected",
                        "contraindicated", "should be held", "risk of",
                        "dangerous", "\"true\""]
    for s in conflict_signals:
        if s in text:
            return True

    danger_words = ["contraindicated", "hold", "discontinue", "risk", "toxicity", "avoid"]
    return any(w in text for w in danger_words)


def call_claude(client, prompt):
    """Call Claude API with retry logic."""
    for attempt in range(3):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=512,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": prompt}],
            )
            return response.content[0].text
        except anthropic.RateLimitError:
            wait = 2 ** (attempt + 1)
            print(f"    Rate limited, waiting {wait}s...")
            time.sleep(wait)
        except anthropic.BadRequestError as e:
            # Check if the 400 error is due to low credits
            if 'credit balance is too low' in str(e).lower():
                print(f"\nFATAL ERROR: Out of API credits. Exiting to prevent infinite retries.")
                sys.exit(1)
            print(f"    Retry {attempt+1}: Bad Request - {e}")
            time.sleep(2)
        except Exception as e:
            print(f"    Retry {attempt+1}: {e}")
            time.sleep(2)
    return "ERROR: All retries failed"


def compute_metrics(results, model_name):
    y_true = np.array([1 if r["gt_conflict"] else 0 for r in results])
    y_pred = np.array([1 if r["pred_conflict"] else 0 for r in results])

    overall = {
        "model": model_name,
        "n": len(results),
        "accuracy": round((y_true == y_pred).mean(), 4),
        "macro_f1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
    }

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    overall.update({"tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)})

    per_stratum = {}
    for s in [0, 1, 2, 3]:
        mask = np.array([r["stratum"] == s for r in results])
        if mask.sum() == 0:
            continue
        s_true = y_true[mask]
        s_pred = y_pred[mask]
        name = {0: "safe", 1: "categorical", 2: "numerical", 3: "multi_hop"}[s]
        m = {"name": name, "n": int(mask.sum()),
             "accuracy": round((s_true == s_pred).mean(), 4)}
        if s == 0:
            m["specificity"] = round(1 - s_pred.mean(), 4)
        else:
            m["detection_rate"] = round(s_pred.mean(), 4)
            m["recall"] = round(recall_score(s_true, s_pred, zero_division=0), 4)
            m["f1"] = round(f1_score(s_true, s_pred, zero_division=0), 4)
        per_stratum[s] = m

    cliff = {}
    if 1 in per_stratum and 2 in per_stratum:
        s1 = per_stratum[1].get("detection_rate", 0)
        s2 = per_stratum[2].get("detection_rate", 0)
        cliff = {
            "s1_categorical": s1,
            "s2_numerical": s2,
            "absolute_drop": round(s1 - s2, 4),
            "relative_drop_pct": round((s1 - s2) / max(s1, 0.001) * 100, 1),
        }
    if 3 in per_stratum:
        cliff["s3_multihop"] = per_stratum[3].get("detection_rate", 0)

    return {"overall": overall, "per_stratum": per_stratum, "cliff": cliff}


def evaluate_claude():
    model_key = "claude-sonnet"

    print("=" * 70)
    print(f"MedConflict-Bench — {MODEL} Evaluation")
    print("=" * 70)

    # Check API key
    if "YOUR-KEY" in ANTHROPIC_API_KEY:
        print("\n  ERROR: Set your ANTHROPIC_API_KEY at the top of the script!")
        print("  Get one at: https://console.anthropic.com/")
        return

    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    # Load benchmark
    examples = [json.loads(l) for l in open(BENCH_PATH, encoding="utf-8")]
    test_examples = [e for e in examples if e.get("split") == "test"]
    print(f"  Test examples: {len(test_examples)}")

    # Per-stratum counts
    for s in [0, 1, 2, 3]:
        n = sum(1 for e in test_examples if e.get("stratum") == s)
        name = {0: "Safe", 1: "Categorical", 2: "Numerical", 3: "Multi-hop"}[s]
        print(f"    S{s} ({name}): {n}")

    # Subsample if MAX_PER_STRATUM is set
    if MAX_PER_STRATUM:
        sampled = []
        for s in [0, 1, 2, 3]:
            stratum_exs = [e for e in test_examples if e.get("stratum") == s]
            sampled.extend(stratum_exs[:MAX_PER_STRATUM])
        test_examples = sampled
        print(f"  Subsampled to {len(test_examples)} (max {MAX_PER_STRATUM}/stratum)")

    # Resume
    ckpt_path = RESULTS_DIR / f"{model_key}_results.jsonl"
    done_ids = set()
    results = []
    if ckpt_path.exists():
        results = [json.loads(l) for l in open(ckpt_path, encoding="utf-8")]
        done_ids = {r["example_id"] for r in results}
        print(f"  Resuming: {len(done_ids)} done")

    remaining = [e for e in test_examples if e.get("example_id") not in done_ids]
    print(f"  To process: {len(remaining)}")

    if not remaining:
        print("  All done! Computing metrics...")
        metrics = compute_metrics(results, model_key)
        for s, data in sorted(metrics["per_stratum"].items()):
            rate_key = "detection_rate" if s > 0 else "specificity"
            print(f"    S{s} ({data['name']}): {rate_key}={data.get(rate_key, '?'):.3f}")
        if metrics["cliff"]:
            print(f"    Cliff: {metrics['cliff'].get('absolute_drop', 0):+.3f}")
        return metrics

    # Process
    print(f"\n  Starting evaluation...")
    t0 = time.time()

    with open(ckpt_path, "a", encoding="utf-8") as fout:
        for i, ex in enumerate(tqdm(remaining, desc=f"  {model_key}")):
            prompt = build_prompt(ex)
            raw = call_claude(client, prompt)
            pred = parse_response(raw)

            result = {
                "example_id": ex["example_id"],
                "stratum": ex["stratum"],
                "gt_conflict": ex.get("conflict_present", False),
                "pred_conflict": pred,
                "raw_response": raw[:500],
                "rule_id": ex.get("rule_id", ""),
            }
            results.append(result)
            fout.write(json.dumps(result, ensure_ascii=False) + "\n")
            fout.flush()

            # Rate limiting: ~1 request per second to stay safe
            time.sleep(0.5)

            if (i + 1) % 100 == 0:
                for s in [1, 2, 3]:
                    s_res = [r for r in results if r["stratum"] == s]
                    if s_res:
                        det = sum(1 for r in s_res if r["pred_conflict"]) / len(s_res)
                        print(f"      S{s}: {det:.3f} ({len(s_res)} done)")

    elapsed = time.time() - t0
    print(f"\n  Completed in {elapsed/60:.1f} min")

    # Final metrics
    metrics = compute_metrics(results, model_key)

    print(f"\n{'=' * 60}")
    print(f"Claude Sonnet RESULTS")
    print(f"{'=' * 60}")
    for s, data in sorted(metrics["per_stratum"].items()):
        rate_key = "detection_rate" if s > 0 else "specificity"
        print(f"  S{s} ({data['name']}): {rate_key}={data.get(rate_key, '?'):.3f} (n={data['n']})")
    if metrics["cliff"]:
        print(f"  Cliff (S1→S2): {metrics['cliff'].get('absolute_drop', 0):+.3f}")
    print(f"  Overall F1:  {metrics['overall']['f1']:.4f}")
    print(f"  Macro-F1:    {metrics['overall']['macro_f1']:.4f}")

    with open(RESULTS_DIR / f"{model_key}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"\n  Saved: {ckpt_path}")

    return metrics


# Run
if __name__ == "__main__":
    evaluate_claude()